In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from gwpy.signal.filter_design import bandpass
from gwpy.timeseries import TimeSeries
from scipy.signal import freqz_zpk

sys.path.append("../src")
from senal import blanquear, ganancia_pasa_banda, pasa_banda, welch

In [ ]:
EVENTO = "GW150914"
DATOS = Path("../datos")
DURACION = 32
DURACION_FFT = 4
BANDA = (35, 350)
CORTE = (30, 400)
RECORTE = (-0.2, 0.1)

In [ ]:
t0 = float((DATOS / f"{EVENTO}_gps.txt").read_text())
datos_h1 = TimeSeries.read(DATOS / f"H1_{EVENTO}_{DURACION}s.hdf5")
datos_l1 = TimeSeries.read(DATOS / f"L1_{EVENTO}_{DURACION}s.hdf5")
fs = int(datos_h1.sample_rate.value)
tiempos = datos_h1.times.value - t0
recorte = (tiempos >= RECORTE[0]) & (tiempos <= RECORTE[1])

In [ ]:
blanqueado_h1 = blanquear(datos_h1.value, fs, DURACION_FFT)
blanqueado_l1 = blanquear(datos_l1.value, fs, DURACION_FFT)

In [ ]:
gwpy_h1 = datos_h1.whiten(fftlength=DURACION_FFT).value
gwpy_l1 = datos_l1.whiten(fftlength=DURACION_FFT).value
print(f"H1: diferencia máxima {np.abs(blanqueado_h1 - gwpy_h1).max():.1e}, valor máximo {np.abs(gwpy_h1).max():.2f}")
print(f"L1: diferencia máxima {np.abs(blanqueado_l1 - gwpy_l1).max():.1e}, valor máximo {np.abs(gwpy_l1).max():.2f}")

In [ ]:
plt.plot(tiempos, blanqueado_h1, label="a mano")
plt.plot(tiempos, blanqueado_h1 - gwpy_h1, label="a mano - gwpy")
plt.xlabel("Tiempo desde t0 (s)")
plt.title("H1 blanqueado")
plt.legend();

In [ ]:
lejos_de_los_bordes = np.abs(tiempos) < 14
frecuencias, psd_blanqueado_h1 = welch(blanqueado_h1[lejos_de_los_bordes], fs, fs)
_, psd_blanqueado_l1 = welch(blanqueado_l1[lejos_de_los_bordes], fs, fs)
plt.semilogy(frecuencias, psd_blanqueado_h1 * fs / 2, label="H1")
plt.semilogy(frecuencias, psd_blanqueado_l1 * fs / 2, label="L1")
plt.axhline(1, color="k")
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("PSD blanqueada × fs / 2")
plt.legend();

In [ ]:
filtrado_h1 = pasa_banda(blanqueado_h1, fs, BANDA, CORTE)
filtrado_l1 = pasa_banda(blanqueado_l1, fs, BANDA, CORTE)
gwpy_filtrado_h1 = datos_h1.whiten(fftlength=DURACION_FFT).bandpass(*BANDA).value
gwpy_filtrado_l1 = datos_l1.whiten(fftlength=DURACION_FFT).bandpass(*BANDA).value
print(f"Correlación con gwpy en el recorte: H1 {np.corrcoef(filtrado_h1[recorte], gwpy_filtrado_h1[recorte])[0, 1]:.3f}, L1 {np.corrcoef(filtrado_l1[recorte], gwpy_filtrado_l1[recorte])[0, 1]:.3f}")

In [ ]:
frecuencias = np.linspace(1, 600, 2400)
_, respuesta = freqz_zpk(*bandpass(*BANDA, fs), worN=frecuencias, fs=fs)
plt.plot(frecuencias, ganancia_pasa_banda(frecuencias, BANDA, CORTE), label="a mano")
plt.plot(frecuencias, np.abs(respuesta) ** 2, label="gwpy (ida y vuelta)")
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("Ganancia")
plt.legend();

In [ ]:
fig, ejes = plt.subplots(2, 1, sharex=True)
for eje, ifo, mio, suyo in zip(ejes, ["H1", "L1"], [filtrado_h1, filtrado_l1], [gwpy_filtrado_h1, gwpy_filtrado_l1]):
    eje.plot(tiempos[recorte], mio[recorte], label="a mano")
    eje.plot(tiempos[recorte], suyo[recorte], label="gwpy")
    eje.set_title(ifo)
    eje.legend()
ejes[1].set_xlabel("Tiempo desde t0 (s)");